# Phase B - Fine-tune `StudentAudio2Landmark` tren MEAD voi vector cam xuc

Nap checkpoint Phase A, fine-tune tren du lieu MEAD da tien xu ly (`phaseB_ready/`), va so sanh
**co / khong co** vector cam xuc tren cac speaker **chua gap** khi train.

| Lan chay | `RUN_NAME` | Noi dung |
| :--- | :--- | :--- |
| B0 | (khong train) | Checkpoint Phase A chay nguyen trang tren MEAD - moc tham chieu |
| B1 | `B1_noemo` | Fine-tune tren MEAD, **khong** co cam xuc (`emotion_dim = 0`) - nhom doi chung |
| B2 | `B2_emo` | Fine-tune tren MEAD, **co** vector cam xuc 7 chieu |

Chi phan chenh giua **B2 va B1** moi la tac dung cua vector cam xuc. B1 va B2 giong het nhau
o moi thu khac (du lieu, so epoch, lr, seed).

## Du lieu (da tien xu ly, xem bao cao Phase A / ghi chu Phase B)

* 41 speaker MEAD, chi goc `front`; chia theo speaker trong `phaseB_ready/split.json`
  (33 train / 4 val / 4 test, seed 0).
* Moi speaker 1 file `<speaker>.npz`: `mel` (80, 62.5 khung/giay, cong thuc AutoVC),
  `lm_res` (do dich so voi mat trung tinh cua speaker, da lam muot Savitzky-Golay 5/2),
  `lm_res_raw` (ban goc, dung de danh gia), `offsets`, `emo`, `lvl`, `ref`, `names`.
* Audio ghep voi landmark **theo thu tu** trong tung nhom (cam xuc, muc) - da kiem chung bang
  do dai va tuong quan nang luong audio / do mo mieng.
* Do lech audio -> mieng do tren MEAD: 8 mel-frame (128 ms), dat o `CFG['audio_lead']`.

## Vector cam xuc (B2)

7 chieu = one-hot cua 7 cam xuc (angry, disgusted, fear, happy, sad, surprised, contempt)
nhan voi `muc / 3`. Neutral = vector 0, nen voi neutral mo hinh B2 tuong duong mo hinh khong
cam xuc.

## Thu tu chay

1. `RUN_NAME = 'B1_noemo'` -> Cell 1 .. 6 (train B1).
2. Doi `RUN_NAME = 'B2_emo'` -> chay lai Cell 1 .. 6 (train B2).
3. Cell 7 (danh gia) - doc lap, tu tim checkpoint B0 / B1 / B2 nao dang co.

Muon chi danh gia: dat `EVAL_ONLY = True` o Cell 1, chay Cell 1, 2, 3 roi nhay toi Cell 7.

## Noi luu

| File | Noi luu |
| :--- | :--- |
| `phaseB_last.pth`, `phaseB_best.pth`, `phaseB_history.json` | `MyDrive/DAT301m/Data_student/student_a2l/phaseB/<RUN_NAME>/` |
| `phaseB_loss_curves.png` | cung thu muc |
| `label_offsets_train.npz` (do lech TB theo nhan, tu speaker train) | `.../student_a2l/phaseB/` |
| `phaseB_evaluation.json` | `.../student_a2l/phaseB/` |

## Cell 1 - Cau hinh

In [ ]:
import os, sys, json, time, math, shutil, subprocess
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

IN_COLAB = 'google.colab' in sys.modules

# ------------------------------------------------------------- CHON LAN CHAY
RUN_NAME    = 'B1_noemo'          # 'B1_noemo' (doi chung)  |  'B2_emo' (co cam xuc)
USE_EMOTION = RUN_NAME.startswith('B2')

# ------------------------------------------------------------------ DUONG DAN
REPO_URL = 'https://github.com/hien087585/EXPRESSIVE-2D-CHARACTER-ANIMATION.git'
REPO_DIR = Path('/content/MakeItTalk')
DRIVE    = Path('/content/drive/MyDrive/DAT301m/Data_student')
_T = os.environ.get('PHASEB_TEST_ROOT')            # chi dung khi chay thu ngoai Colab
if _T:
    REPO_DIR, DRIVE = Path(_T) / 'repo', Path(_T) / 'drive'
READY       = DRIVE / 'Data_emotion_mead_task_b' / 'phaseB_ready'
PHASEA_CKPT = DRIVE / 'student_a2l' / 'phaseA' / '_phaseA_final' / 'phaseA_student_best.pth'
OUT_ROOT    = DRIVE / 'student_a2l' / 'phaseB'
OUT_DIR     = OUT_ROOT / RUN_NAME

EMO_NAMES = ['neutral', 'angry', 'disgusted', 'fear', 'happy', 'sad', 'surprised', 'contempt']
REGIONS   = {'ham': range(0, 17), 'may': range(17, 27), 'mui': range(27, 36),
             'mat': range(36, 48), 'mieng': range(48, 68)}

CFG = dict(
    window            = 18,        # giong Phase A
    audio_lead        = 8,         # mel-frame, do tren MEAD (dinh tuong quan o -8)
    emotion_dim       = 7,
    batch_size        = 256,
    lr                = 1e-4,      # fine-tune: nho hon Phase A (3e-4)
    weight_decay      = 1e-5,
    warmup_epochs     = 1,
    epochs            = 10,
    samples_per_epoch = 200_000,   # cua so 18 frame lay ngau nhien moi epoch
    val_samples       = 20_000,    # co dinh (seed) de so sanh giua cac epoch
    grad_clip         = 1.0,
    amp               = True,
    lambda_dll        = 1.0,
    lambda_dvl        = 0.1,
    lambda_dal        = 0.05,
    eval_stride       = 8,
    seed              = 0,
    auto_resume       = True,
)
# True = chi danh gia (Cell 1 -> 2 -> 3 -> 7), bo qua nap du lieu train/val (tiet kiem ~3 GB RAM)
EVAL_ONLY = False
if os.environ.get('PHASEB_SMOKE'):                 # chay thu nhanh
    CFG.update(epochs=2, samples_per_epoch=2048, val_samples=512, batch_size=64)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('RUN_NAME =', RUN_NAME, '| USE_EMOTION =', USE_EMOTION, '| device =', DEVICE)
if DEVICE.type != 'cuda':
    print('[canh bao] Khong co GPU. Tren Colab: Runtime > Change runtime type > T4 GPU.')

## Cell 2 - Mount Drive, lay repo, kiem tra file

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not (REPO_DIR / 'src' / 'models' / 'model_audio2landmark.py').exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
from src.models.model_audio2landmark import StudentAudio2Landmark, STUDENT_MOUTH_IDX  # noqa: E402

MEAN_STD_PATH = REPO_DIR / 'src' / 'dataset' / 'utils' / 'MEAN_STD_AUTOVC_RETRAIN_MEL_AU.txt'
for p in (READY / 'split.json', PHASEA_CKPT, MEAN_STD_PATH):
    assert p.exists(), 'Thieu file: %s' % p
OUT_DIR.mkdir(parents=True, exist_ok=True)

SPLIT = json.loads((READY / 'split.json').read_text())
for k in ('train', 'val', 'test'):
    missing = [s for s in SPLIT[k] if not (READY / (s + '.npz')).exists()]
    assert not missing, 'Thieu file speaker (%s): %s' % (k, missing)
print('train %d | val %s | test %s' % (len(SPLIT['train']), SPLIT['val'], SPLIT['test']))
print('OUT_DIR =', OUT_DIR)

## Cell 3 - Nap du lieu

Moi clip: `mel[lead:]` ghep voi `landmark[:-lead]` (giong cach dich `AUDIO_LEAD` o Phase A).
Mel duoc chuan hoa bang **dung file mean/std cua Phase A**. Train chi giu ban lam muot (`lm_res`),
val/test giu them ban goc (`lm_res_raw`) de danh gia.

In [ ]:
ms = np.loadtxt(MEAN_STD_PATH, dtype=np.float64).reshape(2, 80)
MEL_MEAN = ms[0].astype(np.float32)
MEL_STD  = np.maximum(ms[1], 1e-5).astype(np.float32)
W, LEAD  = CFG['window'], CFG['audio_lead']


def emo_vec(e, l):
    v = np.zeros(CFG['emotion_dim'], np.float32)
    if e > 0:
        v[e - 1] = l / 3.0
    return v


def load_split(speakers, keep_raw, name):
    t0 = time.time()
    mel_l, res_l, raw_l, clips, refs = [], [], [], [], {}
    pos = 0
    for sp in speakers:
        z = np.load(READY / (sp + '.npz'))
        mel, res, off = z['mel'], z['lm_res'], z['offsets']
        raw = z['lm_res_raw'] if keep_raw else None
        emo, lvl, names = z['emo'], z['lvl'], z['names']
        refs[sp] = z['ref'].astype(np.float32)
        for i in range(len(off) - 1):
            a, b = int(off[i]), int(off[i + 1])
            n = (b - a) - LEAD
            if n < W:
                continue
            m = (mel[a + LEAD:b].astype(np.float32) - MEL_MEAN) / MEL_STD
            mel_l.append(m.astype(np.float16))
            res_l.append(res[a:b - LEAD])
            if keep_raw:
                raw_l.append(raw[a:b - LEAD])
            clips.append(dict(pos=pos, n=n, sp=sp, emo=int(emo[i]), lvl=int(lvl[i]), name=str(names[i])))
            pos += n
    D = dict(name=name, clips=clips, refs=refs,
             mel=np.concatenate(mel_l), res=np.concatenate(res_l))
    if keep_raw:
        D['raw'] = np.concatenate(raw_l)
    D['emo_vec'] = np.stack([emo_vec(c['emo'], c['lvl']) for c in clips])
    n_win = np.array([c['n'] - W + 1 for c in clips])
    D['win_clip'] = np.repeat(np.arange(len(clips)), n_win).astype(np.int32)
    first = np.repeat(np.array([c['pos'] for c in clips]), n_win)
    local = np.arange(n_win.sum()) - np.repeat(np.cumsum(n_win) - n_win, n_win)
    D['win_start'] = (first + local).astype(np.int64)
    n_emo = np.bincount([c['emo'] for c in clips], minlength=8)
    print('%-5s | %2d speaker | %5d clip | %9s frame (%.1f gio) | %9s cua so | %.0f s'
          % (name, len(speakers), len(clips), '{:,}'.format(pos), pos / 62.5 / 3600,
             '{:,}'.format(len(D['win_start'])), time.time() - t0))
    print('        clip theo cam xuc:', dict(zip(EMO_NAMES, n_emo.tolist())))
    return D


if not EVAL_ONLY:
    DTR = load_split(SPLIT['train'], keep_raw=False, name='train')
    DVA = load_split(SPLIT['val'],   keep_raw=False, name='val')
    print('RAM du lieu train: %.2f GB' % ((DTR['mel'].nbytes + DTR['res'].nbytes) / 1e9))
else:
    print('EVAL_ONLY = True -> bo qua du lieu train/val, nhay toi Cell 7.')


def get_batch(D, idx, key='res'):
    s = D['win_start'][idx][:, None] + np.arange(W)
    mel = torch.from_numpy(D['mel'][s].astype(np.float32))
    tgt = torch.from_numpy(D[key][s].astype(np.float32))
    emo = torch.from_numpy(D['emo_vec'][D['win_clip'][idx]])
    return mel, tgt, emo

## Cell 4 - Mo hinh

Nap checkpoint Phase A. Voi B2, `set_emotion_dim(7)` them nhanh cam xuc ma **khong doi dau ra**
luc bat dau (cac cot moi khoi tao bang 0) - cell kiem tra lai dieu nay.

In [ ]:
torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

model = StudentAudio2Landmark.from_checkpoint(str(PHASEA_CKPT), map_location='cpu', eval_mode=False)
if USE_EMOTION:
    model.set_emotion_dim(CFG['emotion_dim'])
model = model.to(DEVICE)

ref_model = StudentAudio2Landmark.from_checkpoint(str(PHASEA_CKPT), map_location='cpu').to(DEVICE)
mel_chk, _, emo_chk = get_batch(DVA, np.arange(min(64, len(DVA['win_start']))))
model.eval()
with torch.no_grad():
    a = model(mel_chk.to(DEVICE), emo_chk.to(DEVICE) if USE_EMOTION else None)
    b = ref_model(mel_chk.to(DEVICE))
print('emotion_dim = %d | tham so = %.3f M | lech so voi Phase A luc bat dau = %.2e (phai ~0)'
      % (model.emotion_dim, model.num_parameters() / 1e6, (a - b).abs().max().item()))
del ref_model

## Cell 5 - Ham loss va danh gia nhanh tren val

Loss giong Phase A: `D-LL + 0.1 D-VL + 0.05 D-A`. `best.pth` chon theo **val LMD toan mat**
(tren landmark da lam muot; danh gia chinh thuc o Cell 7 dung ban goc).

In [ ]:
def compute_loss(pred, gt):
    d_ll = F.mse_loss(pred, gt)
    pv, gv = pred[:, 1:] - pred[:, :-1], gt[:, 1:] - gt[:, :-1]
    d_vl = F.mse_loss(pv, gv)
    d_a = F.mse_loss(pv[:, 1:] - pv[:, :-1], gv[:, 1:] - gv[:, :-1])
    total = CFG['lambda_dll'] * d_ll + CFG['lambda_dvl'] * d_vl + CFG['lambda_dal'] * d_a
    return total, d_ll, d_vl, d_a


def lmd(pred, gt):                          # (B, T, 136) -> (B, T, 68)
    return (pred - gt).reshape(*pred.shape[:2], 68, 2).norm(dim=-1)


VAL_IDX = np.random.default_rng(123).choice(len(DVA['win_start']),
                                            min(CFG['val_samples'], len(DVA['win_start'])), replace=False)


@torch.no_grad()
def validate():
    model.eval()
    s = dict(loss=0.0, d_ll=0.0, lmd=0.0, lmd_mouth=0.0, lmd_emo=0.0, n=0, n_emo=0)
    for i in range(0, len(VAL_IDX), CFG['batch_size']):
        mel, gt, emo = get_batch(DVA, VAL_IDX[i:i + CFG['batch_size']])
        mel, gt, emo = mel.to(DEVICE), gt.to(DEVICE), emo.to(DEVICE)
        pred = model(mel, emo if USE_EMOTION else None).float()
        tot, d_ll, _, _ = compute_loss(pred, gt)
        e = lmd(pred, gt)
        k = len(mel)
        s['loss'] += tot.item() * k; s['d_ll'] += d_ll.item() * k
        s['lmd'] += e.mean(dim=(1, 2)).sum().item()
        s['lmd_mouth'] += e[:, :, 48:].mean(dim=(1, 2)).sum().item()
        is_emo = emo.abs().sum(1) > 0
        s['lmd_emo'] += e.mean(dim=(1, 2))[is_emo].sum().item(); s['n_emo'] += int(is_emo.sum())
        s['n'] += k
    n = s['n']
    return dict(loss=s['loss'] / n, d_ll=s['d_ll'] / n, lmd=s['lmd'] / n,
                lmd_mouth=s['lmd_mouth'] / n, lmd_emo=s['lmd_emo'] / max(s['n_emo'], 1))


v0 = validate()
print('Val TRUOC fine-tune: LMD %.5f | mieng %.5f | clip cam xuc %.5f' % (v0['lmd'], v0['lmd_mouth'], v0['lmd_emo']))

## Cell 6 - Huan luyen (AMP + warmup/cosine + checkpoint len Drive)

`AUTO_RESUME`: Colab bi ngat thi chay lai Cell 1 .. 6, no tu tiep tuc tu `phaseB_last.pth`
**cua dung `RUN_NAME`**. Muon train lai tu dau thi xoa (hoac doi ten) thu muc `OUT_DIR`.

In [ ]:
CKPT_LAST, CKPT_BEST = OUT_DIR / 'phaseB_last.pth', OUT_DIR / 'phaseB_best.pth'
HIST_PATH = OUT_DIR / 'phaseB_history.json'

decay, no_decay = [], []
for n_, p in model.named_parameters():
    (no_decay if p.ndim <= 1 else decay).append(p)
opt = torch.optim.AdamW([{'params': decay, 'weight_decay': CFG['weight_decay']},
                         {'params': no_decay, 'weight_decay': 0.0}], lr=CFG['lr'])
steps_per_epoch = math.ceil(CFG['samples_per_epoch'] / CFG['batch_size'])
total_steps = CFG['epochs'] * steps_per_epoch
warmup = max(1, CFG['warmup_epochs'] * steps_per_epoch)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: (s + 1) / warmup if s < warmup
    else 0.5 * (1 + math.cos(math.pi * min(1.0, (s - warmup) / max(1, total_steps - warmup)))))
use_amp = CFG['amp'] and DEVICE.type == 'cuda'
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

start_epoch, best_lmd, history = 0, float('inf'), []
if CFG['auto_resume'] and CKPT_LAST.exists():
    ck = torch.load(CKPT_LAST, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck['model_state_dict'])
    opt.load_state_dict(ck['optimizer_state_dict'])
    sched.load_state_dict(ck['scheduler_state_dict'])
    start_epoch, best_lmd, history = ck['epoch'] + 1, ck['best_val'], ck.get('history', [])
    print('>> RESUME tu %s (epoch %d xong, best val LMD %.5f)' % (CKPT_LAST.name, ck['epoch'] + 1, best_lmd))

extra = lambda: dict(history=history, phaseB_cfg=CFG, run_name=RUN_NAME, use_emotion=USE_EMOTION,
                     split=SPLIT, phaseA_ckpt=str(PHASEA_CKPT))
rng = np.random.default_rng(CFG['seed'] + start_epoch)
print('%-5s %-10s %-10s %-10s %-10s %-10s %-10s %-10s %s'
      % ('epoch', 'lr', 'loss', 'D-LL', 'val_LMD', 'val_mieng', 'val_camxuc', 'thoi gian', ''))
for epoch in range(start_epoch, CFG['epochs']):
    model.train(); t0 = time.time()
    idx_all = rng.choice(len(DTR['win_start']), steps_per_epoch * CFG['batch_size'], replace=True)
    tl = tdl = 0.0
    for step in range(steps_per_epoch):
        mel, gt, emo = get_batch(DTR, idx_all[step * CFG['batch_size']:(step + 1) * CFG['batch_size']])
        mel, gt, emo = mel.to(DEVICE, non_blocking=True), gt.to(DEVICE, non_blocking=True), emo.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=use_amp):
            pred = model(mel, emo if USE_EMOTION else None)
        loss, d_ll, _, _ = compute_loss(pred.float(), gt)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        scaler.step(opt); scaler.update(); sched.step()
        tl += loss.item(); tdl += d_ll.item()
    v = validate()
    rec = dict(epoch=epoch + 1, lr=opt.param_groups[0]['lr'], train_loss=tl / steps_per_epoch,
               train_d_ll=tdl / steps_per_epoch, seconds=round(time.time() - t0, 1),
               **{'val_' + k: val for k, val in v.items()})
    history.append(rec)
    is_best = v['lmd'] < best_lmd
    if is_best:
        best_lmd = v['lmd']
    model.save_checkpoint(str(CKPT_LAST), optimizer=opt, scheduler=sched, epoch=epoch,
                          best_val=best_lmd, extra=extra())
    if is_best:
        shutil.copy(CKPT_LAST, CKPT_BEST)
    HIST_PATH.write_text(json.dumps(history, indent=1))
    print('%-5d %-10.2e %-10.6f %-10.6f %-10.5f %-10.5f %-10.5f %-10.0f %s'
          % (epoch + 1, rec['lr'], rec['train_loss'], rec['train_d_ll'], v['lmd'], v['lmd_mouth'],
             v['lmd_emo'], rec['seconds'], '<-- BEST' if is_best else ''))
print('Xong. best val LMD = %.5f | %s' % (best_lmd, CKPT_BEST))

## Cell 6b - Duong cong loss (tuy chon)

In [ ]:
import matplotlib
if not IN_COLAB:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
h = json.loads(HIST_PATH.read_text())
ep = [r['epoch'] for r in h]
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
ax[0].plot(ep, [r['train_d_ll'] for r in h], 'o-', label='train'); ax[0].plot(ep, [r['val_d_ll'] for r in h], 'o-', label='val')
ax[0].set_title('D-LL (MSE vi tri)'); ax[0].legend()
ax[1].plot(ep, [r['val_lmd'] for r in h], 'o-', label='toan mat'); ax[1].plot(ep, [r['val_lmd_mouth'] for r in h], 'o-', label='mieng')
ax[1].set_title('val LMD'); ax[1].legend()
ax[2].plot(ep, [r['val_lmd_emo'] for r in h], 'o-'); ax[2].set_title('val LMD - clip co cam xuc')
for a in ax:
    a.set_xlabel('epoch'); a.grid(alpha=.3)
plt.suptitle(RUN_NAME); plt.tight_layout()
plt.savefig(OUT_DIR / 'phaseB_loss_curves.png', dpi=120); plt.show()

## Cell 7 - **Danh gia** tren 4 speaker test (doc lap voi Cell 3 .. 6)

So sanh, tren landmark **goc** (`lm_res_raw`, khong lam muot), tach clip neutral / co cam xuc:

| Ung vien | Y nghia |
| :--- | :--- |
| `dung yen` | luon doan mat trung tinh cua speaker (do dich = 0) |
| `+ lech TB theo nhan` | doan do lech trung binh cua (cam xuc, muc), tinh tu speaker **train** - khong dung audio |
| B0 / B1 / B2 | cac mo hinh |
| `tran` | doan bang trung binh cua chinh clip - gioi han cua phan tinh |

B2 phai **tot hon B1** (tac dung cua cam xuc) va nen **tot hon `+ lech TB theo nhan`**
(neu khong, mo hinh chua lay them duoc gi tu audio ngoai trung binh theo nhan).

In [ ]:
EVAL_RUNS = {
    'B0 Phase A (khong fine-tune)': PHASEA_CKPT,
    'B1 fine-tune, khong cam xuc':  OUT_ROOT / 'B1_noemo' / 'phaseB_best.pth',
    'B2 fine-tune, co cam xuc':     OUT_ROOT / 'B2_emo' / 'phaseB_best.pth',
}
EVAL_RUNS = {k: p for k, p in EVAL_RUNS.items() if Path(p).exists()}
print('Mo hinh se danh gia:', list(EVAL_RUNS))

if 'MEL_MEAN' not in globals():
    ms = np.loadtxt(MEAN_STD_PATH, dtype=np.float64).reshape(2, 80)
    MEL_MEAN, MEL_STD = ms[0].astype(np.float32), np.maximum(ms[1], 1e-5).astype(np.float32)
W, LEAD = CFG['window'], CFG['audio_lead']
if 'load_split' not in globals():
    raise RuntimeError('Chay Cell 3 truoc (dat EVAL_ONLY = True o Cell 1 neu chi muon danh gia).')

# ---- do lech trung binh theo (cam xuc, muc) tu speaker TRAIN (cache tren Drive) ----
OFF_PATH = OUT_ROOT / 'label_offsets_train.npz'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
if OFF_PATH.exists() and json.loads(str(np.load(OFF_PATH)['train']))== SPLIT['train']:
    zo = np.load(OFF_PATH); OFF = {(int(e), int(l)): v for e, l, v in zip(zo['emo'], zo['lvl'], zo['off'])}
else:
    S, C = {}, {}
    for sp in SPLIT['train']:
        z = np.load(READY / (sp + '.npz'))
        x, off, emo, lvl = z['lm_res_raw'], z['offsets'], z['emo'], z['lvl']
        for i in range(len(off) - 1):
            k = (int(emo[i]), int(lvl[i]))
            S[k] = S.get(k, 0) + x[int(off[i]):int(off[i + 1]) - LEAD].astype(np.float64).mean(0)
            C[k] = C.get(k, 0) + 1
    OFF = {k: (S[k] / C[k]).astype(np.float32) for k in S}
    ks = sorted(OFF)
    np.savez(OFF_PATH, emo=np.array([k[0] for k in ks]), lvl=np.array([k[1] for k in ks]),
             off=np.stack([OFF[k] for k in ks]), train=json.dumps(SPLIT['train']))
print('Do lech TB theo nhan: %d nhom (cam xuc, muc)' % len(OFF))

DTE = load_split(SPLIT['test'], keep_raw=True, name='test')
CL = DTE['clips']


@torch.no_grad()
def predict_all(m, D, stride, use_emo, batch=1024):
    starts, cids = [], []
    for ci, c in enumerate(D['clips']):
        st = list(range(0, c['n'] - W + 1, stride))
        if st[-1] != c['n'] - W:
            st.append(c['n'] - W)
        starts += [c['pos'] + s for s in st]; cids += [ci] * len(st)
    starts, cids = np.array(starts), np.array(cids)
    acc = np.zeros((len(D['mel']), 136), np.float64); cnt = np.zeros(len(D['mel']), np.float64)
    for i in range(0, len(starts), batch):
        s = starts[i:i + batch][:, None] + np.arange(W)
        mel = torch.from_numpy(D['mel'][s].astype(np.float32)).to(DEVICE)
        emo = torch.from_numpy(D['emo_vec'][cids[i:i + batch]]).to(DEVICE) if use_emo else None
        out = m(mel, emo).float().cpu().numpy()
        np.add.at(acc, s.ravel(), out.reshape(-1, 136)); np.add.at(cnt, s.ravel(), 1.0)
    return (acc / np.maximum(cnt, 1)[:, None]).astype(np.float32)


def mouth_open(x):
    q = x.reshape(len(x), 68, 2)
    return (np.linalg.norm(q[:, 61] - q[:, 67], axis=1) + np.linalg.norm(q[:, 62] - q[:, 66], axis=1)
            + np.linalg.norm(q[:, 63] - q[:, 65], axis=1)) / 3


def corr(a, b):
    a = a - a.mean(); b = b - b.mean()
    return float((a * b).sum() / (np.sqrt((a * a).sum() * (b * b).sum()) + 1e-12))


preds = {}
for name, path in EVAL_RUNS.items():
    t0 = time.time()
    m = StudentAudio2Landmark.from_checkpoint(str(path), map_location='cpu').to(DEVICE).eval()
    preds[name] = predict_all(m, DTE, CFG['eval_stride'], use_emo=m.emotion_dim > 0)
    print('  du doan xong: %-30s (emotion_dim=%d, %.0f s)' % (name, m.emotion_dim, time.time() - t0))
    del m

CANDS = ['dung yen (mat trung tinh)', '+ lech TB theo nhan (speaker train)'] + list(preds) + ['tran: TB cua chinh clip']
acc = {}; per_sp = {}; mo = {}
for c in CL:
    a, b = c['pos'], c['pos'] + c['n']
    g = DTE['raw'][a:b].astype(np.float32)
    grp = 'neutral' if c['emo'] == 0 else 'cam xuc'
    guess = {CANDS[0]: np.zeros(136, np.float32), CANDS[1]: OFF[(c['emo'], c['lvl'])], CANDS[-1]: g.mean(0)}
    for name in preds:
        guess[name] = preds[name][a:b]
    for name, p in guess.items():
        e = np.linalg.norm((g - p).reshape(len(g), 68, 2), axis=2)
        r = acc.setdefault((grp, name), [np.zeros(68), 0]); r[0] += e.sum(0); r[1] += len(g)
        if grp == 'cam xuc':
            r2 = per_sp.setdefault((c['sp'], name), [0.0, 0]); r2[0] += e.mean(1).sum(); r2[1] += len(g)
    ref = DTE['refs'][c['sp']]
    og = mouth_open(g + ref)
    for name in preds:
        op = mouth_open(preds[name][a:b] + ref)
        mo.setdefault((grp, name), []).append((corr(og, op), op.std() / (og.std() + 1e-12)))

report = dict(split=SPLIT, cfg=CFG, groups={}, per_speaker={}, mouth={})
for grp in ('neutral', 'cam xuc'):
    base = acc[(grp, CANDS[0])][0] / acc[(grp, CANDS[0])][1]
    print('\n== Clip %s (%d speaker test) - %% = tot hon "dung yen" ==' % (grp, len(SPLIT['test'])))
    print('  %-40s %-17s %-17s %s' % ('', 'toan mat', 'mieng', 'theo vung'))
    for name in CANDS:
        m_ = acc[(grp, name)][0] / acc[(grp, name)][1]
        gain = lambda idx: float(100 * (1 - m_[idx].mean() / base[idx].mean()))
        reg = {r: round(gain(list(i)), 1) for r, i in REGIONS.items()}
        print('  %-40s %.4f (%+5.1f%%)  %.4f (%+5.1f%%)  %s'
              % (name, m_.mean(), gain(slice(None)), m_[48:].mean(), gain(slice(48, 68)),
                 ' '.join('%s %+.1f' % kv for kv in reg.items())))
        report['groups'].setdefault(grp, {})[name] = dict(lmd=float(m_.mean()), lmd_mouth=float(m_[48:].mean()),
                                                          gain_pct=round(gain(slice(None)), 2), regions=reg)
    for name in preds:
        cc = np.array(mo[(grp, name)])
        print('  %-40s tuong quan do mo mieng %.3f | ti le bien do %.2f' % (name, cc[:, 0].mean(), cc[:, 1].mean()))
        report['mouth'].setdefault(grp, {})[name] = dict(corr=float(cc[:, 0].mean()), amp=float(cc[:, 1].mean()))

print('\nTung speaker test (clip co cam xuc, toan mat, % tot hon "dung yen"):')
for sp in SPLIT['test']:
    b0 = per_sp[(sp, CANDS[0])]; row = {}
    for name in CANDS[1:]:
        r = per_sp[(sp, name)]; row[name] = round(float(100 * (1 - (r[0] / r[1]) / (b0[0] / b0[1]))), 1)
    report['per_speaker'][sp] = row
    print('  %s: %s' % (sp, ' | '.join('%s %+.1f' % (k.split(' (')[0], v) for k, v in row.items())))

ge = report['groups']['cam xuc']
k1, k2 = 'B1 fine-tune, khong cam xuc', 'B2 fine-tune, co cam xuc'
if k1 in ge and k2 in ge:
    print('\nTAC DUNG CUA CAM XUC (B2 - B1, clip cam xuc, toan mat): %+.1f diem %% | B2 so voi "lech TB theo nhan": %+.1f diem %%'
          % (ge[k2]['gain_pct'] - ge[k1]['gain_pct'], ge[k2]['gain_pct'] - ge[CANDS[1]]['gain_pct']))
(OUT_ROOT / 'phaseB_evaluation.json').write_text(json.dumps(report, indent=1, ensure_ascii=False))
print('Da luu:', OUT_ROOT / 'phaseB_evaluation.json')